# Interactive Meal Planner

## Project Overview

The previous notebook introduced a nutrition-aware recipe recommendation engine capable of recommending recipes that closely match the nutritional requirements of an individual meal.

However, a complete daily diet consists of multiple meals. Recommending each meal independently can result in nutritional imbalances or duplicate recipe suggestions across the day.

This notebook extends the recommendation engine by implementing an interactive meal planning system. The planner generates recommendations one meal at a time, updates the remaining nutritional targets after each selection, prevents duplicate recipe recommendations, and gradually constructs a nutritionally balanced daily meal plan.

## Objectives

The objectives of this notebook are to:

- Generate meal-wise recipe recommendations throughout the day.
- Automatically select the highest-ranked recommendation by default.
- Allow users to choose an alternative recommendation if desired.
- Update the remaining nutritional targets after every meal.
- Prevent duplicate recipe recommendations.
- Construct a complete nutritionally balanced daily meal plan.

In [1]:
from pathlib import Path
import os
import sys

PROJECT_ROOT = Path.cwd().resolve()
if not (PROJECT_ROOT / "data").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "data").exists():
    raise FileNotFoundError("Open this notebook from the NutriCore repository or its notebooks directory.")

NOTEBOOKS_DIR = PROJECT_ROOT / "notebooks"
if str(NOTEBOOKS_DIR) not in sys.path:
    sys.path.insert(0, str(NOTEBOOKS_DIR))
os.chdir(PROJECT_ROOT)

## Meal Planning Workflow

The meal planner follows an iterative recommendation strategy in which recipes are generated one meal at a time rather than planning the entire day simultaneously.

The workflow consists of the following steps:

1. Initialize the user's daily nutritional targets.
2. Generate recipe recommendations for the current meal.
3. Automatically select the highest-ranked recipe or allow the user to choose an alternative.
4. Update the remaining nutritional targets.
5. Exclude the selected recipe from future recommendation rounds.
6. Repeat the process until all meals have been planned.

This iterative approach enables each recommendation to adapt dynamically based on the user's remaining nutritional requirements.

In [2]:
# Imports

import pandas as pd

from nb_utils.recommender_nb import (
    load_dataset,
    generate_recommendations
)

pd.set_option("display.max_columns", None) 
pd.set_option("display.width", None)
pd.set_option("display.expand_frame_repr", False)

## Load Processed Recipe Dataset

The meal planner uses the processed recipe dataset prepared during the previous stages of the project.

This dataset contains all nutritional information and recipe metadata required to generate personalized meal recommendations.

In [3]:
# Load Dataset

recipes_df = load_dataset(
    "data/processed/recipes_processed.parquet"
)

print(f"Recipes Available: {recipes_df.shape[0]:,}")

recipes_df.head()

Recipes Available: 302,965


,RecipeId,Name,Description,RecipeCategory,Keywords,RecipeIngredientParts,Calories,FatContent,CarbohydrateContent,ProteinContent,RecipeServings,DietType,CaloriesPerServing,ProteinPerServing,CarbsPerServing,FatPerServing
0,38.0,Low-Fat Berry Blue Frozen Dessert,Make and share this Low-Fat Berry Blue Frozen ...,Frozen Desserts,"[Dessert, Low Protein, Low Cholesterol, Health...","[blueberries, granulated sugar, vanilla yogurt...",170.9,2.5,37.1,3.2,4.0,Vegetarian,42.725000,0.800000,9.275000,0.625
1,39.0,Biryani,Make and share this Biryani recipe from Food.com.,Chicken Breast,"[Chicken Thigh & Leg, Chicken, Poultry, Meat, ...","[saffron, milk, hot green chili peppers, onion...",1110.7,58.8,84.4,63.4,6.0,Non-Vegetarian,185.116667,10.566667,14.066667,9.800
2,40.0,Best Lemonade,This is from one of my first Good House Keepi...,Beverages,"[Low Protein, Low Cholesterol, Healthy, Summer...","[sugar, lemons, rind of, lemon, zest of, fresh...",311.1,0.2,81.5,0.3,4.0,Vegetarian,77.775000,0.075000,20.375000,0.050
3,41.0,Carina's Tofu-Vegetable Kebabs,This dish is best prepared a day in advance to...,Soy/Tofu,"[Beans, Vegetable, Low Cholesterol, Weeknight,...","[extra firm tofu, eggplant, zucchini, mushroom...",536.1,24.0,64.2,29.3,2.0,Vegetarian,268.050000,14.650000,32.100000,12.000
4,42.0,Cabbage Soup,Make and share this Cabbage Soup recipe from F...,Vegetable,"[Low Protein, Vegan, Low Cholesterol, Healthy,...","[plain tomato juice, cabbage, onion, carrots, ...",103.6,0.4,25.1,4.3,4.0,Vegetarian,25.900000,1.075000,6.275000,0.100


In [4]:
recipes_df.shape

(302965, 16)

## Planner State

The meal planner maintains the current planning state throughout the recommendation process.

The planner continuously tracks:

- Remaining nutritional targets.
- Number of meals yet to be planned.
- Previously selected recipes.
- The complete daily meal plan generated so far.

This information is updated after every meal selection and is used to generate recommendations for subsequent meals.

In [5]:
# Initialize Meal Planner

def initialize_meal_plan(
    calories,
    protein,
    carbs,
    fat,
    meals_per_day
):
    """
    Initialize the meal planner state.
    """

    planner = {
        "remaining_calories": calories,
        "remaining_protein": protein,
        "remaining_carbs": carbs,
        "remaining_fat": fat,
        "remaining_meals": meals_per_day,
        "selected_recipe_ids": [],
        "meals": []
    }

    return planner

### Planner State Update

Both the automatic and interactive meal planners update the planner state after every meal selection.

To avoid duplicating this logic across multiple functions, the planner state update has been encapsulated into a dedicated helper function.

This helper is responsible for:

- Recording the selected recipe.
- Updating the remaining nutritional targets.
- Tracking previously selected recipes.
- Updating the number of meals remaining.

Encapsulating this functionality improves code reusability, readability, and maintainability while ensuring consistent behavior across both meal planning workflows.

In [6]:
# Update Meal Planner

def update_meal_plan(
    planner,
    selected_recipe,
    meal_number
):
    """
    Updates the meal planner after a recipe has been selected.

    Parameters
    ----------
    planner : dict
        Current meal planner state.

    selected_recipe : pd.Series
        Selected recipe.

    meal_number : int
        Current meal number (zero-based).

    Returns
    -------
    dict
        Updated planner state.
    """

    selected_recipe = selected_recipe.copy()

    selected_recipe["Meal"] = meal_number + 1

    planner["meals"].append(
        selected_recipe.to_dict()
    )

    planner["selected_recipe_ids"].append(
        selected_recipe["RecipeId"]
    )

    planner["remaining_calories"] -= (
        selected_recipe["RecommendedCalories"]
    )

    planner["remaining_protein"] -= (
        selected_recipe["RecommendedProtein"]
    )

    planner["remaining_carbs"] -= (
        selected_recipe["RecommendedCarbs"]
    )

    planner["remaining_fat"] -= (
        selected_recipe["RecommendedFat"]
    )

    planner["remaining_meals"] -= 1

    return planner

## Daily Meal Planning Algorithm

The daily meal planner constructs a complete meal plan by repeatedly invoking the recipe recommendation engine.

For each meal, the planner performs the following steps:

1. Generate recipe recommendations using the remaining nutritional targets.
2. Automatically select the highest-ranked recommendation by default.
3. Allow the user to choose an alternative recommendation if desired.
4. Record the selected recipe in the meal plan.
5. Update the remaining nutritional targets.
6. Remove the selected recipe from future recommendation rounds.
7. Repeat the process until all meals have been planned.

By continuously updating the remaining nutritional requirements, each recommendation accounts for the nutritional contribution of previously selected meals while ensuring recipe diversity.

## Recommendation Engine Validation

Before implementing the complete interactive meal planner, the recommendation engine is validated independently using a simplified meal planning function.

This function focuses on generating recommendations for a single meal and displaying the resulting meal plan and remaining nutritional targets. Testing the recommendation engine separately simplifies debugging and confirms that the recommendation logic operates correctly before it is integrated into the complete daily meal planning workflow.

In [7]:
# Daily Meal Planner

def plan_daily_meals(
    recipes_df,
    calories,
    protein,
    carbs,
    fat,
    meals_per_day,
    diet_type="Non-Vegetarian",
    selection_indices=None,
    top_n=10
):
    """
    Generates a complete daily meal plan by repeatedly
    invoking the recipe recommendation engine.
    """

    planner = initialize_meal_plan(
        calories,
        protein,
        carbs,
        fat,
        meals_per_day
    )

    # Validate selection indices.

    if selection_indices is None:

        selection_indices = [0] * meals_per_day

    elif len(selection_indices) < meals_per_day:

        selection_indices.extend(
            [0] * (meals_per_day - len(selection_indices))
        )

    elif len(selection_indices) > meals_per_day:

        raise ValueError(
            "The number of selection indices cannot exceed the number of meals."
        )

    # Build a sequential plan from the remaining targets.

    for meal_number in range(meals_per_day):

        print("\n" + "=" * 70)
        print(f"Meal {meal_number + 1}")
        print("=" * 70)

        available_recipes = recipes_df[
            ~recipes_df["RecipeId"].isin(
                planner["selected_recipe_ids"]
            )
        ]

        recommendations = generate_recommendations(
            df=available_recipes,
            calories=planner["remaining_calories"],
            protein=planner["remaining_protein"],
            carbs=planner["remaining_carbs"],
            fat=planner["remaining_fat"],
            meals_per_day=planner["remaining_meals"],
            diet_type=diet_type,
            top_n=top_n
        )

        selected_index = selection_indices[meal_number]

        if selected_index >= len(recommendations):

            raise ValueError(
                f"Meal {meal_number + 1}: "
                f"Selection index {selected_index} is out of range. "
                f"Only {len(recommendations)} recommendation(s) are available."
            )

        selected_recipe = recommendations.iloc[selected_index].copy()

        print("\n✓ Selected Recipe")
        print("-" * 40)

        print(f"Name      : {selected_recipe['Name']}")
        print(f"Serving   : {selected_recipe['BestServing']}")

        print(
            f"Calories  : {selected_recipe['RecommendedCalories']:.2f} kcal"
        )

        print(
            f"Protein   : {selected_recipe['RecommendedProtein']:.2f} g"
        )

        print(
            f"Carbs     : {selected_recipe['RecommendedCarbs']:.2f} g"
        )

        print(
            f"Fat       : {selected_recipe['RecommendedFat']:.2f} g"
        )

        print(
            f"Distance  : {selected_recipe['Distance']:.4f}"
        )

        planner = update_meal_plan(
            planner,
            selected_recipe,
            meal_number
        )

        print("\nRemaining Nutrition Targets")
        print("-" * 40)

        print(
            f"Calories : {planner['remaining_calories']:.2f} kcal"
        )

        print(
            f"Protein  : {planner['remaining_protein']:.2f} g"
        )

        print(
            f"Carbs    : {planner['remaining_carbs']:.2f} g"
        )

        print(
            f"Fat      : {planner['remaining_fat']:.2f} g"
        )

    planner["meals"] = pd.DataFrame(
        planner["meals"]
    )

    print("\n" + "=" * 70)
    print("DAILY MEAL PLAN GENERATED SUCCESSFULLY")
    print("=" * 70)

    return planner

### Testing the Recommendation Engine

The following examples evaluate the recommendation engine using sample nutritional targets. The objective is to verify that the engine generates nutritionally appropriate recipe recommendations and correctly calculates the remaining nutritional requirements after meal selection.

These examples serve as validation of the recommendation engine prior to its integration into the interactive meal planner.

In [8]:
# Example Meal Plan

meal_plan = plan_daily_meals(
    recipes_df=recipes_df,

    calories=2400,
    protein=160,
    carbs=280,
    fat=70,

    meals_per_day=4,

    diet_type="Non-Vegetarian",

)


Meal 1

✓ Selected Recipe
----------------------------------------
Name      : Vietnamese-Style Chicken Sammies (Banh Mi)
Serving   : 3
Calories  : 596.55 kcal
Protein   : 39.60 g
Carbs     : 68.18 g
Fat       : 17.55 g
Distance  : 0.0101

Remaining Nutrition Targets
----------------------------------------
Calories : 1803.45 kcal
Protein  : 120.40 g
Carbs    : 211.82 g
Fat      : 52.45 g

Meal 2

✓ Selected Recipe
----------------------------------------
Name      : Hot Curry Beef Noodle Bowl
Serving   : 6
Calories  : 596.00 kcal
Protein   : 40.30 g
Carbs     : 73.90 g
Fat       : 17.80 g
Distance  : 0.0139

Remaining Nutrition Targets
----------------------------------------
Calories : 1207.45 kcal
Protein  : 80.10 g
Carbs    : 137.92 g
Fat      : 34.65 g

Meal 3

✓ Selected Recipe
----------------------------------------
Name      : Elizabethan Pork
Serving   : 6
Calories  : 610.50 kcal
Protein   : 39.90 g
Carbs     : 67.30 g
Fat       : 16.90 g
Distance  : 0.0122

Remaining Nutrit

In [9]:
# Display Meal Plan

meal_plan["meals"]

,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore,Meal
0,498633.0,Vietnamese-Style Chicken Sammies (Banh Mi),3,596.55,39.6,68.18,17.55,0.0101,0.9900,1
1,427103.0,Hot Curry Beef Noodle Bowl,6,596.00,40.3,73.90,17.80,0.0139,0.9862,2
2,98837.0,Elizabethan Pork,6,610.50,39.9,67.30,16.90,0.0122,0.9879,3
3,468525.0,Easy Garlic Chicken,4,600.10,39.4,70.90,18.30,0.0148,0.9853,4


In [10]:
# Remaining Nutrition Summary

print("=" * 60)
print("FINAL DAILY NUTRITION SUMMARY")
print("=" * 60)

print(
    f"Remaining Calories : "
    f"{meal_plan['remaining_calories']:.2f} kcal"
)

print(
    f"Remaining Protein  : "
    f"{meal_plan['remaining_protein']:.2f} g"
)

print(
    f"Remaining Carbs    : "
    f"{meal_plan['remaining_carbs']:.2f} g"
)

print(
    f"Remaining Fat      : "
    f"{meal_plan['remaining_fat']:.2f} g"
)

print("=" * 60)

FINAL DAILY NUTRITION SUMMARY
Remaining Calories : -3.15 kcal
Remaining Protein  : 0.80 g
Remaining Carbs    : -0.28 g
Remaining Fat      : -0.55 g


## Interactive Planner Implementation

Following successful validation of the recommendation engine, the complete interactive meal planner is implemented.

Unlike the previous validation function, which demonstrates recommendations for a single meal, the interactive meal planner sequentially generates recommendations for multiple meals throughout the day. After each meal selection, the remaining nutritional targets are updated, previously selected recipes are excluded from future recommendations, and the process continues until the complete daily meal plan has been generated.

This function represents the final implementation of the personalized meal planning system.

In [11]:
# Interactive Meal Planner

def interactive_meal_planner(
    recipes_df,
    calories,
    protein,
    carbs,
    fat,
    meals_per_day,
    diet_type="Non-Vegetarian",
    top_n=10
):

    planner = initialize_meal_plan(
        calories,
        protein,
        carbs,
        fat,
        meals_per_day
    )

    for meal_number in range(meals_per_day):

        print("\n" + "=" * 70)
        print(f"Meal {meal_number + 1}")
        print("=" * 70)

        available_recipes = recipes_df[
            ~recipes_df["RecipeId"].isin(
                planner["selected_recipe_ids"]
            )
        ]

        recommendations = generate_recommendations(
            df=available_recipes,
            calories=planner["remaining_calories"],
            protein=planner["remaining_protein"],
            carbs=planner["remaining_carbs"],
            fat=planner["remaining_fat"],
            meals_per_day=planner["remaining_meals"],
            diet_type=diet_type,
            top_n=top_n
        )

        display_columns = [
            "Name",
            "BestServing",
            "RecommendedCalories",
            "RecommendedProtein",
            "RecommendedCarbs",
            "RecommendedFat",
            "Distance"
        ]

        display_df = (
            recommendations[display_columns]
            .reset_index(drop=True)
        )

        display_df.index += 1
        display_df.index.name = "Choice"

        print("\nTop Recipe Recommendations:\n")
        print(display_df)

        while True:
            choice = input(
                f"\nSelect Recipe (1-{len(display_df)}) "
                "[Press Enter for Recommendation #1]: "
            ).strip()

            if choice == "":  # Default: Top recommendation
                selected_index = 0
                break

            try:
                selected_index = int(choice) - 1

                if 0 <= selected_index < len(display_df):
                    break

                print(
                    f"\nPlease enter a number between "
                    f"1 and {len(display_df)}."
                )

            except ValueError:
                print("\nInvalid input. Please enter a valid number.")

        selected_recipe = recommendations.iloc[selected_index].copy()

        # Display chosen recipe details
        print("\nSelected Recipe")
        print("-" * 40)
        print(f"Name      : {selected_recipe['Name']}")
        print(f"Serving   : {selected_recipe['BestServing']}")
        print(
            f"Calories  : {selected_recipe['RecommendedCalories']:.2f} kcal"
        )
        print(
            f"Protein   : {selected_recipe['RecommendedProtein']:.2f} g"
        )
        print(
            f"Carbs     : {selected_recipe['RecommendedCarbs']:.2f} g"
        )
        print(
            f"Fat       : {selected_recipe['RecommendedFat']:.2f} g"
        )

        # Update planner state
        planner = update_meal_plan(
            planner,
            selected_recipe,
            meal_number
        )

        # Display updated remaining nutrition
        print("\nRemaining Nutrition Targets")
        print("-" * 40)
        print(f"Calories : {planner['remaining_calories']:.2f} kcal")
        print(f"Protein  : {planner['remaining_protein']:.2f} g")
        print(f"Carbs    : {planner['remaining_carbs']:.2f} g")
        print(f"Fat      : {planner['remaining_fat']:.2f} g")

    planner["meals"] = pd.DataFrame(
        planner["meals"]
    )

    return planner

## Interactive Meal Planner Demonstration

The following demonstrations illustrate the complete workflow of the interactive meal planner.

To demonstrate its flexibility, separate meal plans are generated for different dietary preferences. These examples verify that the dietary filtering mechanism developed during preprocessing integrates correctly with the meal planning workflow while maintaining nutritionally balanced recommendations.

### Non-Vegetarian Meal Plan

The demonstration generates a complete daily meal plan using the **Non-Vegetarian** dietary preference. Recipes are selected from the complete dataset while satisfying the specified nutritional targets.

In [12]:
# Run an interactive planner example.

interactive_plan = interactive_meal_planner(
    recipes_df=recipes_df,
    calories=2400,
    protein=160,
    carbs=280,
    fat=70,
    meals_per_day=4,
    diet_type="Non-Vegetarian",
    top_n=10
)


Meal 1

Top Recipe Recommendations:

                                                  Name  BestServing  RecommendedCalories  RecommendedProtein  RecommendedCarbs  RecommendedFat  Distance
Choice                                                                                                                                                  
1           Vietnamese-Style Chicken Sammies (Banh Mi)            3               596.55               39.60             68.18           17.55    0.0101
2                                  Easy Garlic Chicken            4               600.10               39.40             70.90           18.30    0.0148
3                                     Banana Milkshake            1               596.10               39.80             73.50           17.00    0.0157
4                           Hot Curry Beef Noodle Bowl            6               596.00               40.30             73.90           17.80    0.0159
5         Beef Sandwich With Horseradish Bro

In [13]:
# Final Daily Meal Plan

interactive_plan["meals"]


,RecipeId,Name,BestServing,RecommendedCalories,RecommendedProtein,RecommendedCarbs,RecommendedFat,Distance,NutritionScore,Meal
0,468525.0,Easy Garlic Chicken,4,600.10,39.4,70.90,18.30,0.0148,0.9853,1
1,498633.0,Vietnamese-Style Chicken Sammies (Banh Mi),3,596.55,39.6,68.18,17.55,0.0137,0.9864,2
2,359976.0,Chicken and Rice With Spring Vegetables,4,599.70,41.1,68.10,16.90,0.0135,0.9866,3
3,26386.0,Pork Cutlets with Spaghetti,5,626.92,40.0,73.92,17.75,0.0192,0.9810,4


In [14]:
# Remaining Nutrition Summary

print("=" * 60)
print("FINAL DAILY NUTRITION SUMMARY")
print("=" * 60)

print(
    f"Remaining Calories : "
    f"{interactive_plan['remaining_calories']:.2f} kcal"
)

print(
    f"Remaining Protein  : "
    f"{interactive_plan['remaining_protein']:.2f} g"
)

print(
    f"Remaining Carbs    : "
    f"{interactive_plan['remaining_carbs']:.2f} g"
)

print(
    f"Remaining Fat      : "
    f"{interactive_plan['remaining_fat']:.2f} g"
)

print("=" * 60)

FINAL DAILY NUTRITION SUMMARY
Remaining Calories : -23.27 kcal
Remaining Protein  : -0.10 g
Remaining Carbs    : -1.10 g
Remaining Fat      : -0.50 g


## Discussion

The previous demonstrations confirm that the proposed meal planning system successfully generates personalized daily meal plans for given dietary preference.

The recommendation engine was first validated independently before being integrated into the complete interactive meal planner. The final planner continuously updates the remaining nutritional requirements after each meal and prevents duplicate recipe recommendations, enabling the construction of balanced and personalized daily meal plans.

## Summary

In this notebook, an interactive meal planning system was developed by integrating the nutrition-aware recipe recommendation engine with an iterative meal planning workflow.

The planner generates recommendations sequentially, updates the remaining nutritional targets after each meal, and prevents duplicate recipe recommendations throughout the planning process.

This iterative approach enables the construction of nutritionally balanced and personalized daily meal plans while maintaining recipe diversity.

The interactive meal planner developed in this notebook serves as the final component of the hybrid recipe recommendation system by combining nutritional target calculation, recipe recommendation, and sequential meal planning into a complete end-to-end workflow.